#Bidirectional RNNs — Context from Both Directions

In standard RNNs, LSTMs, or GRUs, the model processes text from left to right. However, in language, a word's meaning often depends on words that come after it. For example, in the sentence "The bank of the river," you don't know if "bank" refers to a river or a financial institution until you see the later words. Bidirectional RNNs solve this by training two independent layers—one processing the sequence forward and one backward—and merging their outputs.

1. Understand the Bidirectional Wrapper

In Keras, Bidirectional is a wrapper, not a standalone layer. You can wrap it around any RNN, LSTM, or GRU. It effectively doubles the number of parameters because it creates two copies of the layer.

- **Forward Layer:** Reads: "The" → "bank" → "of" → "the" → "river".

- **Backward Layer:** Reads: "river" → "the" → "of" → "bank" → "The".

2. Build a Bidirectional LSTM Model

We will implement a model that uses bidirectional layers to capture deep contextual relationships.

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

# Model Hyperparameters
vocab_size = 10000
embedding_dim = 128
max_length = 100

model = Sequential([
    # 1. Embedding Layer
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    
    # 2. Bidirectional LSTM Layer
    # We wrap the LSTM layer to process text in both directions
    Bidirectional(LSTM(units=64, return_sequences=True)),
    
    # 3. Second Bidirectional Layer (Stacking)
    Bidirectional(LSTM(units=32)),
    
    # 4. Dense Output
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

c:\Users\md ansar\120-Days-of-ML\ml_env\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

3. nalyze the Output Shape

When you run model.summary(), notice the Output Shape for the Bidirectional layer. If the LSTM units are 64, the output will be 128. This is because the forward and backward outputs are concatenated by default.

In [5]:
# Pass the expected input shape to the model's computation method
# (Batch_size=None, Sequence_Length=max_length)
input_shape = (None, max_length)

for layer in model.layers:
    if "bidirectional" in layer.name:
        # Use compute_output_shape to force Keras to calculate it
        shape = layer.compute_output_shape(input_shape)
        print(f"Layer: {layer.name} | Output Units: {shape[-1]}")

Layer: bidirectional | Output Units: 128
Layer: bidirectional_1 | Output Units: 64
